In [ ]:
import os
import pandas as pd
import sqlalchemy
import numpy as np
from sqlalchemy import text
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 DB 접속 정보 로드

engine = sqlalchemy.create_engine(
    "mysql+pymysql://{user}:{password}@{host}:{port}/{dbname}?charset=utf8".format(
        user     = os.getenv("DB_USER"),
        password = os.getenv("DB_PASSWORD"),
        host     = os.getenv("DB_HOST"),
        port     = os.getenv("DB_PORT"),
        dbname   = os.getenv("DB_NAME"),
    )
)

## 데이터 로드

In [ ]:
# wholesale_orders: 도매 주문 내역
# 정상 주문(status != "") 전체 기간 대상
query = """
SELECT user_id, order_id, product_code, product_name,
       amount, total_price
FROM wholesale_orders
WHERE status != ''
"""
df = pd.read_sql(query, engine)

## STEP 1. 상품별 주문 수량 패턴

In [ ]:
# 상품별 평균/중앙값/최대 주문 수량 — 대량 구매 중심 상품 파악
product_bulk = (
    df.groupby(['product_code', 'product_name'])['amount']
      .agg(['count', 'mean', 'median', 'max'])
      .reset_index()
      .sort_values('mean', ascending=False)
)
product_bulk.head(10)

## STEP 2. 대량 구매(10개 이상) 패턴 분석

In [ ]:
# 10개 이상 주문을 "대량 구매"로 정의
bulk_df = df[df['amount'] >= 10]

In [ ]:
# 대량 구매 빈도 상위 상품
bulk_product = (
    bulk_df.groupby(['product_code', 'product_name'])
           .size()
           .reset_index(name='bulk_count')
           .sort_values('bulk_count', ascending=False)
)
bulk_product.head(10)

In [ ]:
# 대량 구매 빈도 상위 유저
user_bulk = (
    bulk_df.groupby('user_id')
           .size()
           .reset_index(name='bulk_order_cnt')
           .sort_values('bulk_order_cnt', ascending=False)
)
user_bulk.head(10)

In [ ]:
# 전체 주문 대비 대량 구매 비율
bulk_ratio = len(bulk_df) / len(df)
print(f"대량 구매 비율: {bulk_ratio:.1%}")
print(f"전체 주문: {len(df):,}건 | 대량 구매: {len(bulk_df):,}건")